# Harga pangan saat Ramadan: kapan puncak tiap bahan?

Setiap Ramadan orang bilang: "harga pasti naik". Benar, tapi saya penasaran yang lebih spesifik: tiap bahan itu meledaknya kapan? Apakah semua naik ketika puasa dimulai lalu turun setelah Lebaran, atau gelombangnya berbeda-beda per bahan?

In [ ]:
import sys
from pathlib import Path

import json
import numpy as np
import pandas as pd

from tools import gaya

gaya.terapkan()

df = pd.read_csv("data/harga-nasional-mingguan.csv", parse_dates=["tanggal"])
KAL = pd.read_csv("data/tanggal-ramadan.csv", parse_dates=["puasa_mulai", "lebaran"]).set_index("tahun")

print("PIHPS:", df.komoditas.nunique(), "komoditas |", df.tanggal.nunique(), "tanggal |",
      df.tanggal.min().date(), "->", df.tanggal.max().date())
print("kalender pemerintah 8 Ramadan:", KAL.index.min(), "-", KAL.index.max())

PIHPS: 10 komoditas | 144 tanggal | 2019-03-13 -> 2026-04-29
kalender pemerintah 8 Ramadan: 2019 - 2026


## Data & cara ukur

Satu sumber terbuka: PIHPS (bi.go.id/hargapangan) memantau harga eceran sepuluh komoditas pangan strategis di pasar tradisional seluruh provinsi tiap hari kerja. Saya ambil satu snapshot per Rabu di seputar delapan Ramadan (1440-1447 H, 2019-2026); tiap jendela mulai 56 hari sebelum 1 Ramadan sampai 42 hari setelah 1 Syawal, memakai tanggal ketetapan pemerintah yang terverifikasi (`data/tanggal-ramadan.csv`). Supaya tahun demi tahun sejenak, tiap kurva dinormalkan terhadap dasar jendelanya sendiri: rata-rata pekan-pekan yang lebih dari lima pekan sebelum 1 Ramadan. Kalender Muhammadiyah berbeda satu-dua hari di sebagian tahun; edisi ini memakai tanggal pemerintah, dicatat di Batas.

Panen 29 September 2026; kontrak sumber berlanjut dari Edisi 005/007, termasuk catatan bahwa dua belasnya rilis bergeser dan ada bulan kosong server, semuanya dicatat di `data/README.md`.

In [2]:
def kurva(kom, d):
    rows = []
    for th in KAL.index:
        puasa, lebaran = KAL.loc[th, "puasa_mulai"], KAL.loc[th, "lebaran"]
        w = d[(d.tanggal >= puasa - pd.Timedelta(days=56)) &
              (d.tanggal <= lebaran + pd.Timedelta(days=42))].copy()
        if len(w) < 8:
            continue
        w["rel"] = (w.tanggal - puasa).dt.days
        dasar = w.loc[w.rel <= -35, "harga_nasional"]
        if len(dasar) < 3:
            continue
        w["ind"] = w.harga_nasional / dasar.mean()
        w["th"] = th
        rows.append(w[["kom", "th", "rel", "ind"]] if "kom" in w
                    else w.assign(kom=kom)[["kom", "th", "rel", "ind"]])
    return rows

rekam = []
for kom, d in df.groupby("komoditas"):
    rekam += kurva(kom, d[["tanggal", "harga_nasional"]])
IDX = pd.concat(rekam, ignore_index=True)
IDX["bin"] = (IDX.rel // 7) * 7
print("titik kurva:", len(IDX), "| pasangan komoditas-tahun:", IDX.th.nunique(), "x", IDX.kom.nunique())

titik kurva: 1440 | pasangan komoditas-tahun: 8 x 10


## Pembedahan

Gambar pertama: rata-rata lintas delapan tahun untuk tiap bahan, dinormalkan terhadap dasar sebelum Ramadan. Perhatikan puncaknya: tidak semua jatuh di hari yang sama, dan ada dua fase yang terpisah rapi.

In [3]:
def puncak_tahun(kom, d):
    hasil = []
    for th, w in IDX[IDX.kom == kom].groupby("th"):
        puasa, lebaran = KAL.loc[th, "puasa_mulai"], KAL.loc[th, "lebaran"]
        batas = (lebaran - puasa).days + 21
        w2 = w[(w.rel >= -14) & (w.rel <= batas)]
        if len(w2) < 3:
            continue
        p = w2.loc[w2.ind.idxmax()]
        t_naik = w.loc[(w.rel < p.rel) & (w.ind >= 1.05), "rel"].min()
        t_tenang = w.loc[(w.rel > p.rel) & (w.ind <= 1.02), "rel"].min()
        hasil.append({"kom": kom, "th": th, "x_maks": (p.ind - 1) * 100, "t_maks": int(p.rel),
                      "t_naik": float(t_naik) if np.isfinite(t_naik) else float("nan"),
                      "t_tenang": float(t_tenang) if np.isfinite(t_tenang) else float("nan")})
    return hasil


RAMP = []
for kom in sorted(IDX.kom.unique()):
    RAMP += puncak_tahun(kom, IDX)
RAMP = pd.DataFrame(RAMP)
print(RAMP.groupby("kom")[["x_maks", "t_maks", "t_naik", "t_tenang"]].median().round(1))

WAJAH = {
    "Cabai Rawit": (gaya.AKSEN, "-", 2.4, 0.95),
    "Cabai Merah": (gaya.AKSEN, "--", 2.0, 0.90),
    "Bawang Merah": (gaya.INK, "-", 2.0, 0.90),
    "Bawang Putih": (gaya.INK, ":", 1.9, 0.85),
    "Daging Ayam": (gaya.INK, "-.", 1.9, 0.85),
    "Telur Ayam": (gaya.INK, "--", 1.7, 0.80),
    "Beras": (gaya.INK2, "-", 1.1, 0.45),
    "Minyak Goreng": (gaya.INK2, ":", 1.1, 0.45),
    "Gula Pasir": (gaya.INK2, "-.", 1.1, 0.45),
    "Daging Sapi": (gaya.INK2, "--", 1.1, 0.40),
}


def grafik_1(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Gelombang Ramadan dua fase: bumbu dulu, lauk kemudian",
        "Rerata 8 tahun, dinormalkan dasar pra-Ramadan. Cabai memuncak saat puasa baru "
        "mulai; bawang dan lauk justru memuncak menjelang Lebaran; bahan pokok nyaris diam.",
        f"PIHPS nasional, 10 komoditas x {IDX.th.nunique()} Ramadan mingguan (2019-2026)",
        int(len(IDX)),
    )
    for kom in IDX.kom.unique():
        d = IDX[IDX.kom == kom]
        per = d.groupby("bin")["ind"].median()
        warna, ls, lw, al = WAJAH[kom]
        ax.plot(per.index, (per.values - 1) * 100, color=warna, ls=ls, lw=lw, alpha=al,
                zorder=3, label=kom)
    ax.axvspan(0, 30, color=gaya.AKSEN, alpha=0.07, zorder=1)
    ax.axvline(0, color=gaya.INK2, ls=":", lw=0.9)
    ax.text(1.5, ax.get_ylim()[0] * 0.92, "1 Ramadan", fontsize=8.0 * fs,
            color=gaya.INK2, va="bottom")
    ax.axvspan(28, 31, color=gaya.INK, alpha=0.10, zorder=1)
    ax.text(29.5, ax.get_ylim()[1] * 0.97, "Lebaran (beda ±2 hari)", fontsize=8.0 * fs,
            color=gaya.INK2, ha="center", va="top")
    ax.axhline(0, color=gaya.GRID, lw=0.6)
    ax.set_ylabel("perubahan vs dasar pra-Ramadan (%)", fontsize=9.5 * fs)
    ax.set_xlabel("hari terhadap 1 Ramadan (pemerintah)", fontsize=9.5 * fs)
    ax.legend(fontsize=7.4 * fs, loc="lower left", ncol=2, frameon=False,
              handlelength=2.4, columnspacing=1.1, borderaxespad=0.2)
    ax.grid(True, axis="y"); ax.grid(False, axis="x")
    gaya.simpan(fig, "01-gelombang-ramadan", mode)


for mode in gaya.MODE:
    grafik_1(mode)

               x_maks  t_maks  t_naik  t_tenang
kom                                            
Bawang Merah     18.6    41.5   -23.0      52.0
Bawang Putih     10.1    21.0   -11.0      36.0
Beras             2.0    22.5   -16.0      35.0
Cabai Merah      20.7    15.0   -33.5      27.5
Cabai Rawit      22.9     1.5   -32.5      25.5
Daging Ayam      13.7    34.0     0.0      48.0
Daging Sapi       3.0    32.5    24.0      40.5
Gula Pasir        3.9    31.0   -31.5      42.0
Minyak Goreng     1.9    28.0   -11.0      32.0
Telur Ayam        8.3    19.5   -13.0      29.5


Gambar kedua menerjemahkan kurva itu jadi "peta fase": sejak kapan tiap bahan menyala (melewati +5%), di mana puncaknya, dan kapan kembali tenang (≤ +2%). Bar ditata dari yang paling awal menyala. Rentang diwarnai dari rata-rata delapan tahun; mediannya dihitung per bahan dengan langit-langit bootstrap.

In [4]:
from numpy.random import default_rng

rng = default_rng(2026)


def ci_med(x):
    x = pd.Series(x).dropna().values
    if len(x) < 3:
        return np.nan, np.nan, np.nan
    sm = np.array([np.median(rng.choice(x, len(x))) for _ in range(10000)])
    return float(np.median(x)), float(np.percentile(sm, 2.5)), float(np.percentile(sm, 97.5))


RINGKAS = []
for kom, d in RAMP.groupby("kom"):
    xm, lo, hi = ci_med(d.x_maks)
    tn_, tnlo, tnhi = ci_med(d.t_naik)
    tm_ = ci_med(d.t_maks)[0]
    tt_ = ci_med(d.t_tenang)[0]
    RINGKAS.append({"kom": kom, "x_maks": xm, "lo": lo, "hi": hi,
                    "t_naik": tn_, "t_maks": tm_, "t_tenang": tt_,
                    "n": int(d.x_maks.notna().sum()),
                    "n_tenang": int(d.t_tenang.notna().sum())})
RINGKAS = pd.DataFrame(RINGKAS).sort_values("t_naik", ascending=False, na_position="first")


def grafik_2(mode):
    fig, ax, fs = gaya.dasar(
        mode,
        "Urutan nyala per bahan: ramp dimulai sebulan sebelum puasa",
        "Batang dimulai saat bahan melewati +5% dari dasar dan berakhir saat kembali "
        "di bawah +2%; titik: puncak masing-masing. Bahan pokok tak pernah tembus +5%, "
        "yang tergambar titik puncaknya saja. Cabai paling awal dan paling cepat reda.",
        f"Batas median bootstrap 95% dari {int(RAMP.th.nunique())} Ramadan per bahan",
        int(len(IDX)),
    )
    dfu = RINGKAS.iloc[::-1]
    ys = np.arange(len(dfu))
    ax.axvspan(0, 30, color=gaya.AKSEN, alpha=0.07, zorder=1)
    ax.axvspan(28, 31, color=gaya.INK, alpha=0.10, zorder=1)
    ax.text(30.5, len(dfu) - 0.4, "Lebaran", fontsize=8.3 * fs, color=gaya.INK2)
    ax.text(15, -1.0, "Ramadan (pemerintah)", fontsize=8.0 * fs, color=gaya.INK2, ha="center")
    for i, (_, r) in enumerate(dfu.iterrows()):
        x0 = r.t_naik if np.isfinite(r.t_naik) else r.t_maks
        ax.barh(i, r.t_tenang - x0, left=x0, height=0.62,
                color=gaya.AKSEN if "Cabai" in r.kom else gaya.INK, alpha=0.8, zorder=3)
        ax.plot([r.t_maks], [i], marker="D", ms=5, color=gaya.INK2, zorder=4)
        akhir = r.t_tenang
        ax.text(akhir + 2, i, f"tenang hari {akhir:+.0f}", va="center", fontsize=8 * fs, color=gaya.INK2)
    ax.axvline(0, color=gaya.INK2, lw=0.9, ls=":")
    ax.set_yticks(ys)
    ax.set_yticklabels(dfu["kom"], fontsize=9 * fs)
    ax.set_xlim(-42, 60)
    ax.set_xlabel("hari terhadap 1 Ramadan (pemerintah)", fontsize=9.5 * fs)
    ax.grid(True, axis="x"); ax.grid(False, axis="y")
    gaya.simpan(fig, "02-peta-fase-ramadan", mode)


for mode in gaya.MODE:
    grafik_2(mode)

## Temuan

> Gelombang Ramadan itu dua fase, bukan satu: sambal cabai meledak tepat saat puasa dimulai (puncak median hari +2 untuk cabai rawit, +15 untuk cabai merah) dan sudah mereda menjelang Lebaran; bawang, telur, dan ayam baru memuncak menjelang Lebaran (bawang merah di hari +42, ayam +34) dan baru tenang satu-dua pekan sesudahnya. Bahan pokok justru datar: beras, minyak, gula, dan daging sapi puncaknya hanya 2-4%. Dan ramp-nya dimulai sebulan lebih awal dari perkiraan: cabai sudah melewati +5% sekitar hari −32.

In [5]:
metrik = {
    "kalender_pemerintah": KAL.reset_index().assign(
        puasa_mulai=lambda d: d.puasa_mulai.dt.date.astype(str),
        lebaran=lambda d: d.lebaran.dt.date.astype(str)).set_index("tahun_hijri").drop(columns=["tahun"]).to_dict(orient="index"),
    "per_komoditas": RINGKAS.assign(
        x_maks=lambda d: d.x_maks.round(2), lo=lambda d: d.lo.round(2), hi=lambda d: d.hi.round(2),
        t_naik=lambda d: d.t_naik.round(1), t_maks=lambda d: d.t_maks.round(1),
        t_tenang=lambda d: d.t_tenang.round(1))[["kom", "x_maks", "lo", "hi", "t_naik", "t_maks", "t_tenang", "n", "n_tenang"]].to_dict("records"),
    "n_ramadan": int(RAMP.th.nunique()),
    "n_komoditas": int(RAMP.kom.nunique()),
}
print(json.dumps(metrik, indent=2, ensure_ascii=False))
Path("data/metrik-008.json").write_text(json.dumps(metrik, indent=2, ensure_ascii=False))

{
  "kalender_pemerintah": {
    "1440": {
      "puasa_mulai": "2019-05-06",
      "lebaran": "2019-06-05"
    },
    "1441": {
      "puasa_mulai": "2020-04-24",
      "lebaran": "2020-05-24"
    },
    "1442": {
      "puasa_mulai": "2021-04-14",
      "lebaran": "2021-05-13"
    },
    "1443": {
      "puasa_mulai": "2022-04-03",
      "lebaran": "2022-05-02"
    },
    "1444": {
      "puasa_mulai": "2023-03-23",
      "lebaran": "2023-04-22"
    },
    "1445": {
      "puasa_mulai": "2024-03-12",
      "lebaran": "2024-04-10"
    },
    "1446": {
      "puasa_mulai": "2025-03-01",
      "lebaran": "2025-03-31"
    },
    "1447": {
      "puasa_mulai": "2026-02-19",
      "lebaran": "2026-03-21"
    }
  },
  "per_komoditas": [
    {
      "kom": "Beras",
      "x_maks": 1.96,
      "lo": 0.44,
      "hi": 7.62,
      "t_naik": NaN,
      "t_maks": 22.5,
      "t_tenang": 35.0,
      "n": 8,
      "n_tenang": 6
    },
    {
      "kom": "Daging Sapi",
      "x_maks": 2.95,
      "l

2800

## Batas & cara reproduksi

Enam hal perlu dibilang. Pertama, PIHPS memantau sepuluh komoditas di pasar tradisional, bukan harga eceran di semua kanal dan bukan semua bahan. Kedua, "nasional" adalah rerata provinsi dihitung server, bukan rerata tertimbang populasi. Ketiga, tanggal puasa dan Lebaran diambil dari ketetapan pemerintah; penetapan Muhammadiyah berbeda satu-dua hari di sebagian tahun, yang bisa menggeser pembacaan posisi puncak sepekan dari beda dua-tiga minggu bukan hari. Keempat, snapshot Rabu adalah proksi mingguan; bila Rabu tanpa rilis baru, server membalas rilis terakhir, membuat segmen kosong terlihat datar (beberapa segmen 2019/2021/2022 jatuh di fase pasca-Lebaran). Kelima, delapan tahun saja per bahan; sebagian selang keyakinan lebar karena naik-turunnya amat bergantung tahun. Keenam, "puncak" di sini adalah puncak relatif terhadap dasar jendela itu, bukan angka tertinggi sepanjang tahun: di luar Ramadan bahan yang sama juga bisa lebih tinggi.

```bash
python3 tools/data_scraping.py     # PIHPS mingguan 8 Ramadan x 10 komoditas (butuh internet)
python3 -m nbconvert --to notebook --execute --inplace analisis.ipynb
```